# 06 — Holon Subtypes and Classification

The CGA ontology declares six functional holon subtypes and two
lifecycle subtypes. Each carries specific semantics about what
belongs in its four layers. This notebook exercises three subtypes
with dedicated SHACL shapes:

- `cga:AgentHolon` — an active computational agent
- `cga:AggregateHolon` — a materialized view with no original data
- `cga:DataHolon` — a container for domain data with governance

In [ ]:
from holonic import HolonicDataset
import pyshacl

ds = HolonicDataset(load_ontology=True)

def validate(ds, label=""):
    registry = ds.backend.get_graph(ds.registry_iri)
    shapes = ds.backend.get_graph("urn:holonic:ontology:cga-shapes")
    conforms, _, report = pyshacl.validate(
        registry, shacl_graph=shapes, allow_infos=True)
    if label:
        print(f"=== {label} ===")
    print(f"Conforms: {conforms}")
    if not conforms:
        for line in report.split("\n"):
            if "Message" in line or "Severity" in line:
                print(f"  {line.strip()}")
    print()
    return conforms

## AgentHolon — active computational agent

An agent holon represents software that acts on data. Its four
layers have specific meanings: interior = agent state/config,
boundary = output constraints, context = execution history.

In [ ]:
# A well-formed agent holon with all layers
ds.add_holon("urn:holon:email-classifier", "Email Classifier",
             holon_type="cga:AgentHolon")

ds.add_interior("urn:holon:email-classifier", '''
    @prefix schema: <https://schema.org/> .
    <urn:agent:classifier> a <urn:holonic:ontology:AgentHolon> ;
        schema:name "Email Classifier v3" ;
        schema:version "3.1.0" .
    <urn:config:threshold> a schema:PropertyValue ;
        schema:name "spam_threshold" ;
        schema:value "0.85" .
''')

ds.add_boundary("urn:holon:email-classifier", '''
    @prefix sh:  <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    <urn:shapes:ClassificationOutput> a sh:NodeShape ;
        sh:targetClass <urn:Classification> ;
        sh:property [ sh:path <urn:category> ; sh:minCount 1 ; sh:datatype xsd:string ] ;
        sh:property [ sh:path <urn:confidence> ; sh:minCount 1 ; sh:datatype xsd:decimal ] .
''')

ds.add_context("urn:holon:email-classifier", '''
    @prefix prov: <http://www.w3.org/ns/prov#> .
    @prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
    <urn:run:2026-04-22> a prov:Activity ;
        prov:startedAtTime "2026-04-22T10:00:00Z"^^xsd:dateTime ;
        prov:endedAtTime   "2026-04-22T10:05:00Z"^^xsd:dateTime .
''')

validate(ds, "Well-formed AgentHolon")

In [ ]:
# An agent holon missing boundary and context layers
ds.add_holon("urn:holon:bare-agent", "Bare Agent",
             holon_type="cga:AgentHolon")
ds.add_interior("urn:holon:bare-agent", '<urn:state:x> a <urn:AgentState> .')

validate(ds, "AgentHolon missing boundary + context (Info)")

## AggregateHolon — materialized view

An aggregate holon contains no original data. Its interiors are
populated entirely by portal traversals. The shape warns when an
aggregate has data but no provenance linking it to a traversal.

In [ ]:
# Aggregate with traversal provenance (correct)
ds.add_holon("urn:holon:contacts-rollup", "Contacts Rollup",
             holon_type="cga:AggregateHolon")
interior_iri = "urn:holon:contacts-rollup/interior/rollup"
ds.add_interior("urn:holon:contacts-rollup", '''
    @prefix schema: <https://schema.org/> .
    <urn:contact:1> a schema:Person ; schema:name "Aggregated contact" .
''', graph_iri=interior_iri)
ds.add_context("urn:holon:contacts-rollup", f'''
    @prefix prov: <http://www.w3.org/ns/prov#> .
    @prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
    <urn:traversal:001> a prov:Activity ;
        prov:generated <{interior_iri}> ;
        prov:startedAtTime "2026-04-22T12:00:00Z"^^xsd:dateTime .
''')

validate(ds, "AggregateHolon with provenance (correct)")

In [ ]:
# Aggregate with data but NO provenance (suspect)
ds.add_holon("urn:holon:bad-aggregate", "Suspect Aggregate",
             holon_type="cga:AggregateHolon")
ds.add_interior("urn:holon:bad-aggregate", '''
    <urn:raw:data> a <urn:Record> ;
        <urn:field> "This was manually inserted, not traversed" .
''')

validate(ds, "AggregateHolon without provenance (Warning)")

## ClassificationLevel enum

Since 0.4.3, `cga:dataClassification` uses `cga:ClassificationLevel`
individuals (`Public`, `Internal`, `PII`, `Confidential`, `Restricted`) instead
of free-text strings.

In [ ]:
ds.add_holon("urn:holon:customer-data", "Customer Data",
             holon_type="cga:DataHolon")
ds.add_interior("urn:holon:customer-data", '''
    @prefix cga: <urn:holonic:ontology:> .
    <urn:holon:customer-data> cga:dataClassification cga:PII ;
        cga:dataSteward <urn:person:data-lead> .
''')

validate(ds, "DataHolon with PII classification")

## Where to go next

- `05_governed_boundaries` — governance vocabulary in action
- `docs/source/ontology.md` — full class and property reference